In [1]:
from pymongo import MongoClient
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
client = MongoClient('mongodb://localhost:27017')

db = client["sample_supplies"]

collection = db["sales"]

#print(f"total sales: {collection.count_documents({}):,}")

total sales: 5,000


# Q16: How can we calculate the total value of each sales transaction using MongoDB Aggregation?

In [5]:
pipeline = [
    {
        "$unwind": "$items"
    },
    {
        "$group": {
            "_id": "$_id",
            "Store Location": {
                "$first": "$storeLocation"
            },
            "Purchase Method": {
                "$first": "$purchaseMethod"
            },
            "Sale Date": {
                "$first": "$saleDate"
            },
            "Total Transaction Value": {
                "$sum": {
                    "$multiply": [
                        {"$toDouble": "$items.price"},
                        "$items.quantity"
                    ]
                }
            }
        }
    },
    {
        "$sort": {
            "Total Transaction Value": -1
        }
    }
]

result = list(collection.aggregate(pipeline))

result_df = pd.DataFrame(result)

result_df.rename(columns={"_id":"Transaction ID"}, inplace= True)
result_df

,Transaction ID,Store Location,Purchase Method,Sale Date,Total Transaction Value
0,5bd761dcae323e45a93cd1dd,London,Phone,2014-05-17 14:54:13.554,9385.19
1,5bd761ddae323e45a93cd537,London,In store,2017-04-22 22:14:01.622,9105.13
2,5bd761deae323e45a93cde98,New York,In store,2017-06-16 06:52:05.737,9005.51
3,5bd761ddae323e45a93cd810,New York,Online,2014-06-17 16:45:35.954,8998.43
4,5bd761ddae323e45a93cd6fd,Seattle,In store,2014-07-01 05:44:44.358,8989.94
...,...,...,...,...,...
4995,5bd761ddae323e45a93cd69b,Seattle,In store,2017-09-03 13:14:43.271,5.53
4996,5bd761deae323e45a93ce021,Denver,In store,2016-03-18 18:44:36.501,5.49
4997,5bd761ddae323e45a93cd5a1,Denver,In store,2015-12-12 05:14:43.266,5.33
4998,5bd761ddae323e45a93cd814,Denver,In store,2014-03-04 22:07:16.575,5.29
